## IMPORTS

In [ ]:
import warningsimport pandas as pdimport torchfrom torch import nnfrom hazm import Normalizer, word_tokenize, stopwords_listfrom collections import Counterimport stringfrom torch.utils.data import Dataset, DataLoaderfrom sklearn.metrics import precision_score, recall_score, f1_scorefrom sklearn.model_selection import train_test_splitfrom timeit import default_timer as timerimport matplotlib.pyplot as pltfrom tqdm import tqdmimport textwrapwarnings.filterwarnings('ignore')

## Device

In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## Timer

In [3]:
def print_train_time(start: float, end : float, device: torch.device = None):    total_time = end - start    print(f"Train time on {device}: {total_time/60:.3f} minutes\n\n")    return total_time

## DATA

In [4]:
emails = pd.read_csv("data/emails.csv")

In [5]:
emails.head()

,text,label
0,﻿ممنون آقا سامان.\nمن پارسال اصلا آزاد شرکت نک...,ham
1,﻿سلام آقای کریمی\nبالاخره آزمونارشد تموم شد من...,ham
2,﻿درود بر حاج وحیدی بنده بعنوان یک دکتری تاریخ ...,ham
3,﻿با سلام و احترام\nضمن تقدیر از مسولین محترم ...,ham
4,﻿با سلام اینجانب یک دستگاه خودرو پراید 131 با ...,ham


## Q1: Preprocessing

In [ ]:
# Hazm componentsnormalizer = Normalizer()stopwords = set(stopwords_list())punctuations = string.punctuation + "،؛؟«»"def preprocess_persian_text(text):    # Normalize    text = normalizer.normalize(text).replace('\ufeff', '').replace('\u200c', '')    # Tokenize and clean    tokens = [        ''.join(c for c in token if c not in punctuations)        for token in word_tokenize(text)    ]    # Remove stopwords and empty tokens    return [t for t in tokens if t and t not in stopwords]# Apply preprocessingemails['processed_text_tokens'] = emails['text'].apply(preprocess_persian_text)emails['processed_text_joined'] = emails['processed_text_tokens'].apply(' '.join)for _, row in emails.head().iterrows():    print(f"Original: {row['text']}")    print(f"Processed Tokens: {row['processed_text_tokens']}")    print(f"Processed Joined: {row['processed_text_joined']}\n")

Original: ﻿ممنون آقا سامان.
من پارسال اصلا آزاد شرکت نکرده بودم و سراسری هم قبول نشدم. فقط میخواستم بدونم شرایط چطوریه واسه سال بعد. مرسی از راهنمایی هاتون.
Processed Tokens: ['ممنون', 'آقا', 'سامان', 'پارسال', 'اصلا', 'آزاد', 'شرکت', 'نکردهبودم', 'سراسری', 'قبول', 'نشدم', 'میخواستم', 'بدونم', 'شرایط', 'چطوریه', 'واسه', 'سال', 'مرسی', 'راهنمایی', 'هاتون']
Processed Joined: ممنون آقا سامان پارسال اصلا آزاد شرکت نکردهبودم سراسری قبول نشدم میخواستم بدونم شرایط چطوریه واسه سال مرسی راهنمایی هاتون

Original: ﻿سلام آقای کریمی
بالاخره آزمونارشد تموم شد من راحت شدم
یکم راهنمایی می خوام
بهترین مجله ی تخصصی زبان شناسی که بتونم اشتراک بگیرم کدومه؟
و در زمینه ی جامعه شناسی و روانشناسی زبان کتابای خوب و مرجع رو لطفا بهم معرفی کنید. صرفا به خاطر علاقه ی خودم می خوام مطالعه ی آزاد و تخصصی داشته باشم برای کنکور نیست
مرسی
دوستان دیگه هم اگه کمک کنند ممنونمی شم 
Processed Tokens: ['سلام', 'کریمی', 'بالاخره', 'آزمونارشد', 'تموم', 'راحت', 'شدم', 'یکم', 'راهنمایی', 'میخوام', 'مجلهی', 'تخصصی', 'زبانشناسی', 

In [7]:
PAD = "<PAD>"  # PaddingUNK = "<UNK>"  # Unknown

In [ ]:
# Flatten all tokenized text into one listall_tokens = [token for tokens in emails['processed_text_tokens'] for token in tokens]# Count token frequenciestoken_freq = Counter(all_tokens)# Set vocab, starting with PAD and UNKvocab = [PAD, UNK] + sorted(token_freq.keys())# Map tokens to indicestoken2idx = {token: idx for idx, token in enumerate(vocab)}# Vocab sizevocab_size = len(token2idx)print(f"Vocabulary size: {vocab_size}")

Vocabulary size: 27769


In [9]:
token2idx

{'<PAD>': 0,
 '<UNK>': 1,
 'A': 2,
 'AAAAAAAAAAM': 3,
 'AAC': 4,
 'AB': 5,
 'AC': 6,
 'ACS': 7,
 'ACTIONALLPRODUCTSBRANDF٪D': 8,
 'ACTIONOFFICE': 9,
 'ADSL': 10,
 'AFxigJ': 11,
 'AG': 12,
 'AGetyourPassport': 13,
 'AIDS': 14,
 'AIFF': 15,
 'AL': 16,
 'AM': 17,
 'AMAD': 18,
 'AMD': 19,
 'AMPANG': 20,
 'ANDOR': 21,
 'AP': 22,
 'APPLE': 23,
 'APPS': 24,
 'APPlE': 25,
 'ARM': 26,
 'ARMANI': 27,
 'ASCE': 28,
 'ASME': 29,
 'ASP': 30,
 'ASTM': 31,
 'ATI': 32,
 'ATM': 33,
 'AUX': 34,
 'AVENUE': 35,
 'AVG': 36,
 'AWSurveys': 37,
 'AX': 38,
 'AXRKhQvBZNpeK': 39,
 'Ab': 40,
 'Abbas': 41,
 'Abstract': 42,
 'Academic': 43,
 'Academy': 44,
 'Accelerator': 45,
 'Account': 46,
 'Accountکلیک': 47,
 'Acer': 48,
 'Acrobat': 49,
 'Active': 50,
 'AdSense': 51,
 'Address': 52,
 'Address٪': 53,
 'Administration': 54,
 'Adobe': 55,
 'Adobeمنتشر': 56,
 'Ads': 57,
 'AdsAffiliate': 58,
 'AdsID': 59,
 'AdsIDadstypepercent': 60,
 'AdsIDadstypepercentname٪D': 61,
 'AdsIDadstypepercentname٪DA٪A': 62,
 'AdsIDadstypep

In [ ]:
max_len = 50def tokens_to_indices(tokens, token2idx, max_len):    # Convert tokens to indices    indices = [token2idx.get(token, token2idx["<UNK>"]) for token in tokens]    # Pad or cut short    if len(indices) < max_len:        indices += [token2idx["<PAD>"]] * (max_len - len(indices))    else:        indices = indices[:max_len]    return indices# Apply conversionemails['input_ids'] = emails['processed_text_tokens'].apply(    lambda tokens: tokens_to_indices(tokens, token2idx, max_len))label_map = {"spam": 1, "ham": 0}emails['label'] = emails['label'].map(label_map)

filling missing values in labels

In [ ]:
print(emails['label'].isna().sum())  # Show how many NaNs exist

0
0


## Split

In [ ]:
# First split: train (70%)train_df, temp_df = train_test_split(    emails, test_size=0.3, stratify=emails['label'], random_state=42)# Then split temp: val (10%) vs test (20%)val_df, test_df = train_test_split(    temp_df, test_size=2/3, stratify=temp_df['label'], random_state=42)

## DataLoaders 

In [13]:
class EmailDataset(Dataset):    def __init__(self, df):        self.inputs = df['input_ids'].tolist()        self.labels = df['label'].tolist()    def __len__(self):        return len(self.inputs)    def __getitem__(self, idx):        x = torch.tensor(self.inputs[idx], dtype=torch.long)        y = torch.tensor(self.labels[idx], dtype=torch.float)        return x, y

In [14]:
# Create datasetstrain_dataset = EmailDataset(train_df)val_dataset = EmailDataset(val_df)test_dataset = EmailDataset(test_df)# Create data loadersbatch_size = 64train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)val_loader   = DataLoader(val_dataset, batch_size=batch_size)test_loader  = DataLoader(test_dataset, batch_size=batch_size)

## Q2: A simple model including:

1. an embedding layer
2. a recurrent layer
3. a fully connected layer

plus different metrics and loss plots

In [15]:
class SimpleRNNClassifier(nn.Module):    def __init__(self, vocab_size, embed_dim=100, hidden_dim=64):        super(SimpleRNNClassifier, self).__init__()        self.embedding = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embed_dim, padding_idx=token2idx["<PAD>"])        self.rnn = nn.RNN(input_size=embed_dim, hidden_size=hidden_dim, batch_first=True)        self.fc = nn.Linear(hidden_dim, 1)    def forward(self, x):        embedded = self.embedding(x)                # [batch_size, seq_len, embed_dim]        _, hidden = self.rnn(embedded)              # hidden: [1, batch_size, hidden_dim]        out = self.fc(hidden.squeeze(0))            # [batch_size, 1]        return out.squeeze(1)                       # [batch_size]